# Phase 8: Model Evaluation + Phase 9: Threshold Optimization

This notebook delivers **Phase 8** and **Phase 9** together.

- **Phase 8** runs the *first and only* evaluation on the held-out **test set** (untouched since the Phase 3 split) to get an honest, final read on how the champion model performs, and confirms whether LightGBM alone or the Hybrid Risk Score (Phase 7) should go forward.
- **Phase 9** finds a better decision threshold than the default 0.5 — using the **validation set** to search, then confirming the chosen threshold once on the test set. Three lenses are used: F1-optimal, business-target (recall/precision constraints), and cost-based (using real transaction amounts).

---
## Phase 8, Step 1: Setup — Load Test Set & Final Models

The test set has not been touched since the Phase 3 split — every prior comparison (Phase 5 supervised models, Phase 6 unsupervised, Phase 7 hybrid) was decided on the **validation** set. This is the first time test data is used, so these numbers are the honest, final estimate of real-world performance.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              average_precision_score, confusion_matrix, classification_report,
                              roc_curve, precision_recall_curve)

# Load processed splits
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Load the final artifacts from Phases 5-7
best_supervised_model = joblib.load('../models/best_supervised_model.pkl')  # LightGBM
iso_forest = joblib.load('../models/isolation_forest.pkl')
hybrid_config = joblib.load('../models/hybrid_config.pkl')

print('Model loaded:', type(best_supervised_model).__name__)
print('Hybrid config:', hybrid_config)
print('\nTest set shape:', X_test.shape, ' Fraud cases in test set:', int(y_test.sum()))


## Phase 8, Step 2: Evaluate the Supervised Champion (LightGBM) on the Test Set

In [ ]:
y_proba_test_sup = best_supervised_model.predict_proba(X_test)[:, 1]
y_pred_test_sup = (y_proba_test_sup >= 0.5).astype(int)

print('=== LightGBM — Test Set Performance (threshold = 0.5) ===\n')
print(classification_report(y_test, y_pred_test_sup, target_names=['Legit', 'Fraud'], digits=4))

sup_test_metrics = {
    'Precision': precision_score(y_test, y_pred_test_sup),
    'Recall': recall_score(y_test, y_pred_test_sup),
    'F1': f1_score(y_test, y_pred_test_sup),
    'ROC-AUC': roc_auc_score(y_test, y_proba_test_sup),
    'PR-AUC': average_precision_score(y_test, y_proba_test_sup)
}
print('Summary:', {k: round(v, 4) for k, v in sup_test_metrics.items()})
print('\nConfusion Matrix (rows=actual, cols=predicted):')
print(confusion_matrix(y_test, y_pred_test_sup))


## Phase 8, Step 3: Recompute the Hybrid Risk Score on the Test Set

The Isolation Forest and the min/max used for normalization were both fit on **training/validation** data only (saved in `hybrid_config.pkl`), so we reuse those exact values here rather than re-fitting anything on test data. Test-set anomaly scores are clipped to `[0, 1]` since a handful of test points can fall slightly outside the validation min/max range.

In [ ]:
iso_scores_test = -iso_forest.score_samples(X_test)  # higher = more anomalous
anomaly_score_test = (iso_scores_test - hybrid_config['iso_score_min']) / (
    hybrid_config['iso_score_max'] - hybrid_config['iso_score_min'])
anomaly_score_test = np.clip(anomaly_score_test, 0, 1)

SUPERVISED_WEIGHT = hybrid_config['supervised_weight']
ANOMALY_WEIGHT = hybrid_config['anomaly_weight']

risk_score_test = SUPERVISED_WEIGHT * y_proba_test_sup + ANOMALY_WEIGHT * anomaly_score_test
risk_score_test_100 = (risk_score_test * 100).round(2)
hybrid_pred_test = (risk_score_test >= 0.5).astype(int)

print('=== Hybrid Model (0.7 * LightGBM + 0.3 * Isolation Forest) — Test Set ===\n')
hybrid_test_metrics = {
    'Precision': precision_score(y_test, hybrid_pred_test, zero_division=0),
    'Recall': recall_score(y_test, hybrid_pred_test),
    'F1': f1_score(y_test, hybrid_pred_test),
    'ROC-AUC': roc_auc_score(y_test, risk_score_test),
    'PR-AUC': average_precision_score(y_test, risk_score_test)
}
print({k: round(v, 4) for k, v in hybrid_test_metrics.items()})
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, hybrid_pred_test))


## Phase 8, Step 4: Final Comparison — Champion Decision

This confirms, on data neither model has ever seen or been tuned on, whether the Phase 7 finding (hybrid underperforms LightGBM alone) still holds.

In [ ]:
final_comparison_df = pd.DataFrame({
    'LightGBM (alone)': sup_test_metrics,
    'Hybrid (0.7 sup + 0.3 anomaly)': hybrid_test_metrics
}).T.round(4)

print('=== Final Test-Set Comparison ===\n')
print(final_comparison_df)

champion_name = final_comparison_df['PR-AUC'].idxmax()
print(f"\nChampion model (by test PR-AUC): {champion_name}")

if champion_name == 'LightGBM (alone)':
    champion_proba_test = y_proba_test_sup
    print('This confirms the Phase 7 finding: the anomaly score dilutes the supervised '
          'signal rather than improving it. LightGBM alone moves forward as the champion '
          'model for Phase 9 (threshold optimization) and beyond. The Isolation Forest / '
          'hybrid score are kept as a secondary signal — useful for flagging patterns the '
          'supervised model has never seen, even though it does not top the retrospective metric.')
else:
    champion_proba_test = risk_score_test
    print('The hybrid score wins on the test set — proceeding with it as champion.')

final_comparison_df.to_csv('../models/final_test_evaluation.csv')
print("\nSaved models/final_test_evaluation.csv")


## Phase 8, Step 5: ROC & Precision-Recall Curves

In [ ]:
fpr_sup, tpr_sup, _ = roc_curve(y_test, y_proba_test_sup)
fpr_hyb, tpr_hyb, _ = roc_curve(y_test, risk_score_test)
prec_sup, rec_sup, _ = precision_recall_curve(y_test, y_proba_test_sup)
prec_hyb, rec_hyb, _ = precision_recall_curve(y_test, risk_score_test)
baseline_pr = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].plot(fpr_sup, tpr_sup, label=f"LightGBM (AUC={sup_test_metrics['ROC-AUC']:.4f})")
axes[0].plot(fpr_hyb, tpr_hyb, label=f"Hybrid (AUC={hybrid_test_metrics['ROC-AUC']:.4f})")
axes[0].plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random guess')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Test Set'); axes[0].legend()

axes[1].plot(rec_sup, prec_sup, label=f"LightGBM (PR-AUC={sup_test_metrics['PR-AUC']:.4f})")
axes[1].plot(rec_hyb, prec_hyb, label=f"Hybrid (PR-AUC={hybrid_test_metrics['PR-AUC']:.4f})")
axes[1].axhline(baseline_pr, linestyle='--', color='grey', label=f'No-skill baseline ({baseline_pr:.4f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — Test Set'); axes[1].legend()

plt.tight_layout()
plt.show()


## Phase 8, Step 6: Confusion Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_test_sup), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Legit', 'Pred Fraud'], yticklabels=['Actual Legit', 'Actual Fraud'], ax=axes[0])
axes[0].set_title('LightGBM (threshold=0.5)')

sns.heatmap(confusion_matrix(y_test, hybrid_pred_test), annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Pred Legit', 'Pred Fraud'], yticklabels=['Actual Legit', 'Actual Fraud'], ax=axes[1])
axes[1].set_title('Hybrid (threshold=0.5)')

plt.tight_layout()
plt.show()


## Phase 8, Step 7: Error Analysis — What Is the Champion Model Missing?

To see the real cost of errors, we recover the **original, unscaled transaction Amount** for the test set. It was dropped after scaling in Phase 3 preprocessing and never saved separately, so it's recovered by re-running the exact same dedup + `train_test_split` (same `random_state=42`) against `data/creditcard.csv` — this reproduces the identical test rows in the identical order, verified by shape and index alignment.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df_raw = pd.read_csv('../data/creditcard.csv').drop_duplicates()
raw_amount_all = df_raw['Amount'].copy()

scaler_amt, scaler_time = StandardScaler(), StandardScaler()
df_raw['Amount_scaled'] = scaler_amt.fit_transform(df_raw[['Amount']])
df_raw['Time_scaled'] = scaler_time.fit_transform(df_raw[['Time']])
df_raw2 = df_raw.drop(['Amount', 'Time'], axis=1)

X_all = df_raw2.drop('Class', axis=1)
y_all = df_raw2['Class']
X_train_r, X_temp_r, y_train_r, y_temp_r = train_test_split(
    X_all, y_all, test_size=0.30, random_state=42, stratify=y_all)
X_val_r, X_test_r, y_val_r, y_test_r = train_test_split(
    X_temp_r, y_temp_r, test_size=0.50, random_state=42, stratify=y_temp_r)

assert np.allclose(X_test_r.reset_index(drop=True).values, X_test.values), 'Split mismatch!'
amount_test = raw_amount_all.loc[X_test_r.index].reset_index(drop=True)
print('Raw test-set Amount recovered and verified aligned with X_test/y_test.')

fn_mask = (y_pred_test_sup == 0) & (y_test.values == 1)   # missed fraud
fp_mask = (y_pred_test_sup == 1) & (y_test.values == 0)   # false alarm

print(f"\nMissed fraud (False Negatives): {fn_mask.sum()} transactions, "
      f"${amount_test[fn_mask].sum():,.2f} total value missed")
print(f"Caught fraud (True Positives):  {int(((y_pred_test_sup==1)&(y_test.values==1)).sum())} transactions")
print(f"False alarms (False Positives): {fp_mask.sum()} legitimate transactions flagged")
print(f"\nModel confidence on the missed-fraud cases (should be just under 0.5 if 'close calls'):")
print(pd.Series(y_proba_test_sup[fn_mask]).describe()[['mean','min','50%','max']])


---
# Phase 9: Threshold Optimization

## Phase 9, Step 1: Why 0.5 Isn't the Right Threshold

The default `0.5` cutoff has no special meaning for a problem this imbalanced (~0.17% fraud rate). It was never chosen — it's just what `predict()` uses by default. The right threshold depends on the trade-off the business actually cares about:
- Missing a fraud (False Negative) costs roughly the transaction amount.
- A false alarm (False Positive) costs a manual review, which is much cheaper but not free.

All threshold search below is done on the **validation set** — the test set is touched only once at the end, to report an unbiased final number for the threshold that gets chosen.

## Phase 9, Step 2: Precision / Recall / F1 vs. Threshold

In [ ]:
y_proba_val_sup = best_supervised_model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.01, 1.00, 0.01)
prec_list, rec_list, f1_list = [], [], []
for t in thresholds:
    pred = (y_proba_val_sup >= t).astype(int)
    prec_list.append(precision_score(y_val, pred, zero_division=0))
    rec_list.append(recall_score(y_val, pred, zero_division=0))
    f1_list.append(f1_score(y_val, pred, zero_division=0))

threshold_sweep_df = pd.DataFrame({
    'threshold': thresholds, 'precision': prec_list, 'recall': rec_list, 'f1': f1_list
})

plt.figure(figsize=(10, 5.5))
plt.plot(thresholds, prec_list, label='Precision')
plt.plot(thresholds, rec_list, label='Recall')
plt.plot(thresholds, f1_list, label='F1', linewidth=2)
plt.axvline(0.5, linestyle='--', color='grey', label='Default (0.5)')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs. Threshold (Validation Set)')
plt.legend()
plt.tight_layout()
plt.show()

f1_optimal_idx = int(np.argmax(f1_list))
f1_optimal_threshold = thresholds[f1_optimal_idx]
print(f"F1-optimal threshold: {f1_optimal_threshold:.2f}  "
      f"(Precision={prec_list[f1_optimal_idx]:.4f}, Recall={rec_list[f1_optimal_idx]:.4f}, "
      f"F1={f1_list[f1_optimal_idx]:.4f})")


## Phase 9, Step 3: Business-Target Thresholds

Two common real-world asks: *"catch almost all the fraud"* vs. *"don't annoy too many legitimate customers."* For each, we pick the threshold that best satisfies the constraint while keeping the other metric as high as possible.

In [ ]:
TARGET_RECALL = 0.90
TARGET_PRECISION = 0.90

# Highest threshold that still achieves at least TARGET_RECALL (maximizes precision under the constraint)
recall_ok = threshold_sweep_df[threshold_sweep_df['recall'] >= TARGET_RECALL]
recall_target_reachable = len(recall_ok) > 0
high_recall_threshold = recall_ok['threshold'].max() if recall_target_reachable else thresholds[0]

# Lowest threshold that still achieves at least TARGET_PRECISION (maximizes recall under the constraint)
precision_ok = threshold_sweep_df[threshold_sweep_df['precision'] >= TARGET_PRECISION]
precision_target_reachable = len(precision_ok) > 0
high_precision_threshold = precision_ok['threshold'].min() if precision_target_reachable else thresholds[-1]

def report_threshold(name, t):
    row = threshold_sweep_df.iloc[(threshold_sweep_df['threshold'] - t).abs().idxmin()]
    print(f"{name}: threshold={t:.2f} -> Precision={row['precision']:.4f}, "
          f"Recall={row['recall']:.4f}, F1={row['f1']:.4f}")

print(f'--- Targeting Recall >= {TARGET_RECALL:.0%} (catch almost all fraud) ---')
if recall_target_reachable:
    report_threshold('High-recall', high_recall_threshold)
else:
    best_recall = threshold_sweep_df['recall'].max()
    print(f"No threshold reaches {TARGET_RECALL:.0%} recall on the validation set "
          f"(best achievable recall is {best_recall:.4f}, at threshold {thresholds[0]:.2f}). "
          f"Showing the lowest threshold instead, which gets closest:")
    report_threshold('Closest to high-recall', high_recall_threshold)

print(f'\n--- Targeting Precision >= {TARGET_PRECISION:.0%} (minimize false alarms) ---')
if precision_target_reachable:
    report_threshold('High-precision', high_precision_threshold)
else:
    best_precision = threshold_sweep_df['precision'].max()
    print(f"No threshold reaches {TARGET_PRECISION:.0%} precision on the validation set "
          f"(best achievable precision is {best_precision:.4f}). Showing the highest "
          f"threshold instead, which gets closest:")
    report_threshold('Closest to high-precision', high_precision_threshold)


## Phase 9, Step 4: Cost-Based Threshold Optimization

The most business-realistic approach: assign a real dollar cost to each error type and pick the threshold that minimizes total expected cost.
- **Missed fraud (FN)** costs the full transaction amount (assumes the loss is unrecovered).
- **False alarm (FP)** costs a fixed manual-review fee.

Raw amounts are recovered the same way as Phase 8, Step 7, but for the validation set.

In [ ]:
amount_val = raw_amount_all.loc[X_val_r.index].reset_index(drop=True)
assert np.allclose(X_val_r.reset_index(drop=True).values, X_val.values), 'Split mismatch!'

REVIEW_COST = 10  # assumed cost of a manual fraud-review, in dollars — adjust to your real ops cost

costs = []
for t in thresholds:
    pred = (y_proba_val_sup >= t).astype(int)
    fn_mask_v = (pred == 0) & (y_val.values == 1)
    fp_mask_v = (pred == 1) & (y_val.values == 0)
    total_cost = amount_val[fn_mask_v].sum() + fp_mask_v.sum() * REVIEW_COST
    costs.append(total_cost)

cost_optimal_idx = int(np.argmin(costs))
cost_optimal_threshold = thresholds[cost_optimal_idx]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, costs)
plt.axvline(0.5, linestyle='--', color='grey', label='Default (0.5)')
plt.axvline(cost_optimal_threshold, linestyle='--', color='green', label=f'Cost-optimal ({cost_optimal_threshold:.2f})')
plt.xlabel('Threshold'); plt.ylabel('Total Estimated Cost ($)')
plt.title(f'Total Cost vs. Threshold (Validation Set, review cost=${REVIEW_COST})')
plt.legend()
plt.tight_layout()
plt.show()

default_cost = costs[int(np.argmin(np.abs(thresholds - 0.5)))]
print(f"Cost-optimal threshold: {cost_optimal_threshold:.2f}  -> total cost ${costs[cost_optimal_idx]:,.2f}")
print(f"Cost at default 0.5 threshold: ${default_cost:,.2f}")
print(f"Estimated savings from re-thresholding: ${default_cost - costs[cost_optimal_idx]:,.2f} "
      f"on this validation set alone")


## Phase 9, Step 5: Compare All Candidate Thresholds, Then Confirm Once on the Test Set

Everything above was decided on the validation set. The chosen threshold is now applied to the test set **exactly once**, to report the final, unbiased number.

In [ ]:
def metrics_at(y_true, y_proba, t):
    pred = (y_proba >= t).astype(int)
    return {
        'threshold': round(t, 2),
        'Precision': round(precision_score(y_true, pred, zero_division=0), 4),
        'Recall': round(recall_score(y_true, pred, zero_division=0), 4),
        'F1': round(f1_score(y_true, pred, zero_division=0), 4)
    }

candidates = {
    'Default': 0.5,
    'F1-optimal': f1_optimal_threshold,
    f'High-recall (>= {TARGET_RECALL:.0%})': high_recall_threshold,
    f'High-precision (>= {TARGET_PRECISION:.0%})': high_precision_threshold,
    'Cost-optimal': cost_optimal_threshold
}

threshold_comparison_df = pd.DataFrame(
    {name: metrics_at(y_val, y_proba_val_sup, t) for name, t in candidates.items()}
).T
print('=== Candidate Thresholds — Validation Set ===\n')
print(threshold_comparison_df)

# --- Final decision ---
# Cost-optimal balances both error types using real business economics rather than an
# abstract statistical criterion, so it is the recommended default. Swap FINAL_THRESHOLD
# below if your business instead needs a hard recall or precision guarantee.
FINAL_THRESHOLD = round(float(cost_optimal_threshold), 2)
print(f"\n>>> Selected final threshold: {FINAL_THRESHOLD} <<<")

test_metrics_final = metrics_at(y_test, y_proba_test_sup, FINAL_THRESHOLD)
print('\n=== Final Threshold — ONE-TIME Test Set Confirmation ===')
print(test_metrics_final)
print('\nConfusion Matrix at final threshold (test set):')
print(confusion_matrix(y_test, (y_proba_test_sup >= FINAL_THRESHOLD).astype(int)))


## Phase 9, Step 6: Save Final Artifacts for Phase 10 Onward

In [ ]:
threshold_config = {
    'final_threshold': FINAL_THRESHOLD,
    'method': 'cost-optimal',
    'review_cost_assumed': REVIEW_COST,
    'candidates': {name: round(float(t), 2) for name, t in candidates.items()},
    'validation_comparison': threshold_comparison_df.to_dict(orient='index'),
    'test_set_confirmation': test_metrics_final
}
joblib.dump(threshold_config, '../models/threshold_config.pkl')
threshold_comparison_df.to_csv('../models/threshold_optimization_comparison.csv')

print('Saved models/threshold_config.pkl  (final threshold + all candidate thresholds)')
print('Saved models/threshold_optimization_comparison.csv')
print('\nPhase 8 & 9 complete.')
print(f"Champion model: LightGBM  |  Final decision threshold: {FINAL_THRESHOLD}")
print('Ready for Phase 10: Explainable AI (SHAP).')
